In [42]:
import pandas as pd
import numpy as np

from pymongo import MongoClient
from dotenv import load_dotenv
import os

from sklearn.model_selection import train_test_split

In [43]:
load_dotenv()

MONGODB_URI = os.getenv("MONGODB_URI")

if not MONGODB_URI:
    raise RuntimeError("MONGODB_URI not found.")

In [44]:
client = MongoClient(MONGODB_URI)

db = client["aqi_predictor"]

collection = db["aqi_features"]

df = pd.DataFrame(list(collection.find()))

df.head()

,_id,city,timestamp,hour,day,month,day_of_week,aqi,aqi_change_rate,pm25,pm10,o3,no2,so2,co,temperature,humidity,pressure,wind_speed
0,6a72e460fb5e2680d90bbb0a,karachi,2025-08-05T08:00:00+00:00,8,5,8,1,48,0.0,11.49,45.48,44.97,0.05,0.31,76.93,29.8,70,1002.5,20.9
1,6a72e460fb5e2680d90bbb0b,karachi,2025-08-05T09:00:00+00:00,9,5,8,1,47,-1.0,11.23,44.70,44.90,0.05,0.31,76.80,29.5,71,1002.2,20.0
2,6a72e460fb5e2680d90bbb0c,karachi,2025-08-05T10:00:00+00:00,10,5,8,1,46,-1.0,10.95,44.54,44.71,0.05,0.31,76.69,29.3,72,1002.0,19.5
3,6a72e461fb5e2680d90bbb0d,karachi,2025-08-05T11:00:00+00:00,11,5,8,1,44,-2.0,10.50,44.20,44.39,0.06,0.31,77.06,29.2,72,1001.6,19.9
4,6a72e461fb5e2680d90bbb0e,karachi,2025-08-05T12:00:00+00:00,12,5,8,1,43,-1.0,10.21,44.55,43.94,0.06,0.31,77.63,28.9,73,1001.4,19.5


In [45]:
if "_id" in df.columns:
    df.drop(columns="_id", inplace=True)

df.head()

,city,timestamp,hour,day,month,day_of_week,aqi,aqi_change_rate,pm25,pm10,o3,no2,so2,co,temperature,humidity,pressure,wind_speed
0,karachi,2025-08-05T08:00:00+00:00,8,5,8,1,48,0.0,11.49,45.48,44.97,0.05,0.31,76.93,29.8,70,1002.5,20.9
1,karachi,2025-08-05T09:00:00+00:00,9,5,8,1,47,-1.0,11.23,44.70,44.90,0.05,0.31,76.80,29.5,71,1002.2,20.0
2,karachi,2025-08-05T10:00:00+00:00,10,5,8,1,46,-1.0,10.95,44.54,44.71,0.05,0.31,76.69,29.3,72,1002.0,19.5
3,karachi,2025-08-05T11:00:00+00:00,11,5,8,1,44,-2.0,10.50,44.20,44.39,0.06,0.31,77.06,29.2,72,1001.6,19.9
4,karachi,2025-08-05T12:00:00+00:00,12,5,8,1,43,-1.0,10.21,44.55,43.94,0.06,0.31,77.63,28.9,73,1001.4,19.5


In [46]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

df = df.sort_values("timestamp")

df.reset_index(drop=True, inplace=True)

df.head()

,city,timestamp,hour,day,month,day_of_week,aqi,aqi_change_rate,pm25,pm10,o3,no2,so2,co,temperature,humidity,pressure,wind_speed
0,karachi,2025-08-05 08:00:00+00:00,8,5,8,1,48,0.0,11.49,45.48,44.97,0.05,0.31,76.93,29.8,70,1002.5,20.9
1,karachi,2025-08-05 09:00:00+00:00,9,5,8,1,47,-1.0,11.23,44.70,44.90,0.05,0.31,76.80,29.5,71,1002.2,20.0
2,karachi,2025-08-05 10:00:00+00:00,10,5,8,1,46,-1.0,10.95,44.54,44.71,0.05,0.31,76.69,29.3,72,1002.0,19.5
3,karachi,2025-08-05 11:00:00+00:00,11,5,8,1,44,-2.0,10.50,44.20,44.39,0.06,0.31,77.06,29.2,72,1001.6,19.9
4,karachi,2025-08-05 12:00:00+00:00,12,5,8,1,43,-1.0,10.21,44.55,43.94,0.06,0.31,77.63,28.9,73,1001.4,19.5


In [47]:
FORECAST_HOURS = 72

df["target_aqi"] = df["aqi"].shift(-FORECAST_HOURS)

df.head()

,city,timestamp,hour,day,month,day_of_week,aqi,aqi_change_rate,pm25,pm10,o3,no2,so2,co,temperature,humidity,pressure,wind_speed,target_aqi
0,karachi,2025-08-05 08:00:00+00:00,8,5,8,1,48,0.0,11.49,45.48,44.97,0.05,0.31,76.93,29.8,70,1002.5,20.9,53.0
1,karachi,2025-08-05 09:00:00+00:00,9,5,8,1,47,-1.0,11.23,44.70,44.90,0.05,0.31,76.80,29.5,71,1002.2,20.0,54.0
2,karachi,2025-08-05 10:00:00+00:00,10,5,8,1,46,-1.0,10.95,44.54,44.71,0.05,0.31,76.69,29.3,72,1002.0,19.5,54.0
3,karachi,2025-08-05 11:00:00+00:00,11,5,8,1,44,-2.0,10.50,44.20,44.39,0.06,0.31,77.06,29.2,72,1001.6,19.9,53.0
4,karachi,2025-08-05 12:00:00+00:00,12,5,8,1,43,-1.0,10.21,44.55,43.94,0.06,0.31,77.63,28.9,73,1001.4,19.5,52.0


In [48]:
df = df.dropna(subset=["target_aqi"])

df.reset_index(drop=True, inplace=True)

df.tail()

,city,timestamp,hour,day,month,day_of_week,aqi,aqi_change_rate,pm25,pm10,o3,no2,so2,co,temperature,humidity,pressure,wind_speed,target_aqi
8419,karachi,2026-08-02 03:00:00+00:00,3,2,8,6,79,-1.0,25.49,112.59,35.13,0.13,0.55,69.17,28.4,81,1000.1,10.7,76.0
8420,karachi,2026-08-02 04:00:00+00:00,4,2,8,6,78,-1.0,25.04,112.68,35.10,0.13,0.53,68.97,28.6,79,1000.5,11.6,77.0
8421,karachi,2026-08-02 05:00:00+00:00,5,2,8,6,78,0.0,24.90,114.16,34.60,0.13,0.53,68.77,29.2,76,1000.6,11.3,77.0
8422,karachi,2026-08-02 06:00:00+00:00,6,2,8,6,78,0.0,24.82,115.22,34.26,0.12,0.53,68.72,28.9,74,1001.5,10.5,77.0
8423,karachi,2026-08-02 07:00:00+00:00,7,2,8,6,77,-1.0,24.56,115.12,34.30,0.10,0.52,68.80,28.9,75,1001.3,12.2,77.0


In [49]:
print(df.shape)

df.info()

(8424, 19)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8424 entries, 0 to 8423
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   city             8424 non-null   object             
 1   timestamp        8424 non-null   datetime64[ns, UTC]
 2   hour             8424 non-null   int64              
 3   day              8424 non-null   int64              
 4   month            8424 non-null   int64              
 5   day_of_week      8424 non-null   int64              
 6   aqi              8424 non-null   int64              
 7   aqi_change_rate  8424 non-null   float64            
 8   pm25             8424 non-null   float64            
 9   pm10             8424 non-null   float64            
 10  o3               8424 non-null   float64            
 11  no2              8424 non-null   float64            
 12  so2              8424 non-null   float64            
 13  co     

In [50]:
FEATURES = [
    "hour",
    "day",
    "month",
    "day_of_week",
    "pm25",
    "pm10",
    "o3",
    "no2",
    "so2",
    "co",
    "temperature",
    "humidity",
    "pressure",
    "wind_speed",
]

In [51]:
X = df[FEATURES]

y = df["target_aqi"]

print(X.shape)
print(y.shape)

(8424, 14)
(8424,)


In [52]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False,
)

In [53]:
print("Training")

print(X_train.shape)
print(y_train.shape)

print()

print("Testing")

print(X_test.shape)
print(y_test.shape)

Training
(6739, 14)
(6739,)

Testing
(1685, 14)
(1685,)


In [54]:
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import numpy as np

In [55]:
model = LinearRegression()

model.fit(X_train, y_train)

print("Linear Regression model trained successfully.")

Linear Regression model trained successfully.


In [56]:
y_pred = model.predict(X_test)

print(y_pred[:10])

[68.21844907 69.68066393 69.89839121 68.66987558 67.38518705 65.49909248
 63.15625674 61.19404833 59.08925062 57.67387668]


In [57]:
mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")

MAE  : 22.81
RMSE : 28.74
R²   : -1.3672


In [58]:
results = X_test.copy()

results["Actual AQI"] = y_test.values

results["Predicted AQI"] = y_pred

results.head(20)

,hour,day,month,day_of_week,pm25,pm10,o3,no2,so2,co,temperature,humidity,pressure,wind_speed,Actual AQI,Predicted AQI
6739,3,23,5,5,19.74,120.68,55.17,0.06,0.25,86.72,29.4,73,1008.2,9.9,55.0,68.218449
6740,4,23,5,5,19.81,121.42,55.58,0.05,0.24,86.77,30.6,66,1008.7,10.7,55.0,69.680664
6741,5,23,5,5,19.79,121.44,55.35,0.04,0.23,86.89,31.6,62,1008.9,12.0,55.0,69.898391
6742,6,23,5,5,19.71,121.00,54.30,0.04,0.21,86.80,32.3,60,1008.6,12.8,56.0,68.669876
6743,7,23,5,5,19.50,119.74,52.70,0.03,0.19,86.68,32.6,58,1008.3,13.4,57.0,67.385187
6744,8,23,5,5,19.09,116.89,50.91,0.03,0.18,86.35,32.5,58,1007.8,14.4,57.0,65.499092
6745,9,23,5,5,18.48,112.57,49.20,0.03,0.17,86.26,32.0,62,1007.3,15.5,58.0,63.156257
6746,10,23,5,5,17.76,107.44,47.70,0.03,0.16,86.17,31.5,65,1006.9,16.4,58.0,61.194048
6747,11,23,5,5,16.91,101.73,46.14,0.03,0.16,85.86,31.0,67,1006.4,16.7,58.0,59.089251
6748,12,23,5,5,15.97,95.49,45.03,0.03,0.16,85.54,30.9,67,1006.2,15.8,57.0,57.673877


In [59]:
importance = pd.DataFrame({
    "Feature": FEATURES,
    "Coefficient": model.coef_
})

importance = importance.sort_values(
    by="Coefficient",
    key=abs,
    ascending=False,
)

importance

,Feature,Coefficient
2,month,2.327119
8,so2,-2.174105
12,pressure,1.776649
10,temperature,-0.753498
6,o3,0.500617
7,no2,0.442016
1,day,0.348273
4,pm25,0.234456
11,humidity,-0.171172
3,day_of_week,0.125016


In [60]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(model, "../models/linear_regression.pkl")

print("Model saved successfully.")

Model saved successfully.


Linear Regression

In [61]:
from sklearn.ensemble import RandomForestRegressor

In [62]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [63]:
rf_predictions = rf_model.predict(X_test)

rf_predictions[:10]

array([52.76477717, 51.84408187, 52.84021707, 54.29124152, 58.24786308,
       59.08197848, 58.64925391, 58.09317731, 58.31278264, 53.31149707])

In [64]:
rf_mae = mean_absolute_error(y_test, rf_predictions)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))

rf_r2 = r2_score(y_test, rf_predictions)

print(f"MAE  : {rf_mae:.2f}")
print(f"RMSE : {rf_rmse:.2f}")
print(f"R²   : {rf_r2:.4f}")

MAE  : 16.01
RMSE : 22.06
R²   : -0.3945


In [65]:
rf_results = pd.DataFrame({
    "Actual AQI": y_test.values,
    "Predicted AQI": rf_predictions
})

rf_results.head(20)

,Actual AQI,Predicted AQI
0,55.0,52.764777
1,55.0,51.844082
2,55.0,52.840217
3,56.0,54.291242
4,57.0,58.247863
5,57.0,59.081978
6,58.0,58.649254
7,58.0,58.093177
8,58.0,58.312783
9,57.0,53.311497


In [66]:
importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": rf_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance

,Feature,Importance
6,o3,0.374636
1,day,0.151575
2,month,0.149253
9,co,0.096982
4,pm25,0.060070
3,day_of_week,0.037870
5,pm10,0.037229
8,so2,0.031248
12,pressure,0.024345
7,no2,0.011044


In [67]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(
    rf_model,
    "../models/random_forest.pkl"
)

print("Random Forest model saved successfully.")

Random Forest model saved successfully.


In [68]:
comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest"
    ],
    "MAE": [
        mae,
        rf_mae
    ],
    "RMSE": [
        rmse,
        rf_rmse
    ],
    "R²": [
        r2,
        rf_r2
    ]
})

comparison

,Model,MAE,RMSE,R²
0,Linear Regression,22.810726,28.741618,-1.367193
1,Random Forest,16.014742,22.059584,-0.394459
